# 01. Pipeline de Datos
## TFM — Estimación de Precios de la Electricidad y Arbitraje Energético
### Jaime Cremades Castelló | CUNEF Universidad | 2026

Este notebook cubre tres etapas del procesamiento de datos:

1. **Descarga y construcción del dataset** — precio OMIE, demanda REE/ESIOS, generación, variables meteorológicas, gas y CO₂
2. **Variables previstas** — descarga de previsiones PBF eólica, solar, temperatura y viento desde ESIOS y Open-Meteo
3. **Preprocesamiento final** — limpieza, imputación de nulos, split temporal y guardado del dataset listo para modelado

**Input:** APIs de OMIE, REE/ESIOS, Open-Meteo e Investing.com
**Output:** `dataset_final_clean.csv` — 26 variables, enero 2020 – febrero 2026


---

# PARTE 1 — Descarga y construcción del dataset

---

# Pipeline Para crear el dataset

## Librerias

In [1]:
import pandas as pd
from datetime import datetime, timedelta
import os

import requests
import zipfile
import io

import numpy as np

import holidays

## Logica de actualizacion del dataset

Este bloque de código se encarga de gestionar automáticamente la descarga de datos en función del estado actual del dataset, permitiendo un funcionamiento incremental eficiente.

### 🟢 Caso 1: Primera ejecución (no existe dataset)

Si el archivo `dataset_final.csv` no existe:

- Se asume que es la primera vez que se ejecuta el pipeline.
- Se define el rango completo de descarga:
  - Inicio: **1 de enero de 2020**
  - Fin: **día de ayer**
- No hay datos previos (`df_existing = None`).

👉 Resultado: se descargan todos los datos históricos.

---

### 🔵 Caso 2: Modo incremental (dataset ya existe)

Si el dataset ya existe:

- Se carga el dataset actual.
- Se obtiene la última fecha disponible (`last_date`).
- Se define el nuevo rango de descarga:
  - Inicio: **día siguiente al último dato disponible**
  - Fin: **día de ayer**

👉 De esta forma solo se descargan los datos que faltan.

---

### 🔴 Control de actualización

Se comprueba si realmente hay datos nuevos:

- Si `start_date > end_date`:
  - Significa que el dataset ya está actualizado.
  - Se detiene la ejecución del pipeline mediante `SystemExit`.

👉 Esto evita llamadas innecesarias a APIs y reduce el tiempo de ejecución.

---

In [2]:
DATASET_FINAL_PATH = "dataset_final.csv"

today = datetime.today().date()
yesterday = today - timedelta(days=1)

# CASO 1: NO EXISTE DATASET

if not os.path.exists(DATASET_FINAL_PATH):
    
    print("🟢 Primera ejecución → descargar todo")
    
    start_date = datetime(2020, 1, 1).date()
    end_date = yesterday
    
    df_existing = None

# CASO 2: YA EXISTE DATASET

else:
    
    print("🔵 Modo incremental")
    
    df_existing = pd.read_csv(DATASET_FINAL_PATH, parse_dates=["datetime"])
    
    last_date = df_existing["datetime"].max().date()
    
    start_date = last_date - timedelta(days=45) # Buffer de 31 dias por esios
    end_date = yesterday

    print("Actualizar:", start_date, "→", end_date)

    if start_date > end_date:
        raise SystemExit("✅ Dataset ya actualizado")

🔵 Modo incremental
Actualizar: 2026-02-08 → 2026-04-14


## OMIE

Datos omie - se usa web scrapping en bucles para descargar los datoy y meterlos en una carpeta
Se sacan datos horarios del precio de la electricidad desde 2020 - hoy

In [3]:
def descargar_omie_zip(year, output_folder):
    
    url = f"https://www.omie.es/es/file-download?parents=marginalpdbc&filename=marginalpdbc_{year}.zip"
    
    print(f"⬇️ Descargando ZIP {year}...")
    
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"❌ Error ZIP {year}")
        return
    
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        
        for file_name in z.namelist():
            
            # solo archivos de precio
            if file_name.endswith(".1"):
                
                output_path = os.path.join(output_folder, os.path.basename(file_name))
                
                # evitar sobreescribir si ya existe
                if not os.path.exists(output_path):
                    with open(output_path, "wb") as f:
                        f.write(z.read(file_name))
        
    print(f"✅ ZIP {year} procesado")

output_folder = "omie_data"
os.makedirs(output_folder, exist_ok=True)

# ZIP solo primera vez
if df_existing is None and len(os.listdir(output_folder)) == 0:
    for year in [2020, 2021, 2022]:
        descargar_omie_zip(year, output_folder)

start_date = datetime.combine(start_date, datetime.min.time())
end_date = datetime.combine(end_date, datetime.min.time())

current_date = start_date

while current_date <= end_date:
    date_str = current_date.strftime("%Y%m%d")
    filename = f"marginalpdbc_{date_str}.1"
    
    file_path = os.path.join(output_folder, filename)

    if os.path.exists(file_path):
        print(f"{filename} ya existe")
        current_date += timedelta(days=1)
        continue
    
    url = f"https://www.omie.es/es/file-download?parents=marginalpdbc&filename={filename}"
    
    print(f"Descargando {filename}...")
    
    response = requests.get(url)
    
    if response.status_code == 200 and len(response.content) > 100:
        with open(file_path, "wb") as f:
            f.write(response.content)
        print("OK")
    else:
        print("No disponible")
    
    current_date += timedelta(days=1)

marginalpdbc_20260208.1 ya existe
marginalpdbc_20260209.1 ya existe
marginalpdbc_20260210.1 ya existe
marginalpdbc_20260211.1 ya existe
marginalpdbc_20260212.1 ya existe
marginalpdbc_20260213.1 ya existe
marginalpdbc_20260214.1 ya existe
marginalpdbc_20260215.1 ya existe
marginalpdbc_20260216.1 ya existe
marginalpdbc_20260217.1 ya existe
marginalpdbc_20260218.1 ya existe
marginalpdbc_20260219.1 ya existe
marginalpdbc_20260220.1 ya existe
marginalpdbc_20260221.1 ya existe
marginalpdbc_20260222.1 ya existe
marginalpdbc_20260223.1 ya existe
marginalpdbc_20260224.1 ya existe
marginalpdbc_20260225.1 ya existe
marginalpdbc_20260226.1 ya existe
marginalpdbc_20260227.1 ya existe
marginalpdbc_20260228.1 ya existe
marginalpdbc_20260301.1 ya existe
marginalpdbc_20260302.1 ya existe
marginalpdbc_20260303.1 ya existe
marginalpdbc_20260304.1 ya existe
marginalpdbc_20260305.1 ya existe
marginalpdbc_20260306.1 ya existe
marginalpdbc_20260307.1 ya existe
marginalpdbc_20260308.1 ya existe
marginalpdbc_2

Aqui se juntan todos los archivos sacados de omie en uno ara crear un dataset de solo precios

In [4]:
data_folder = "omie_data"
all_data = []

for file in os.listdir(data_folder):
    if file.endswith(".1"):
        file_path = os.path.join(data_folder, file)

        try:
            df = pd.read_csv(
                file_path,
                sep=";",
                header=None,
                skiprows=1,
                engine="python"
            )

            # Eliminar columnas basura extra
            df = df.iloc[:, :6]

            df.columns = ["year", "month", "day", "period", "price_spain", "price_portugal"]

            # Eliminar filas tipo "*"
            df = df[pd.to_numeric(df["year"], errors="coerce").notna()]

            # Convertir tipos
            df["year"] = df["year"].astype(int)
            df["month"] = df["month"].astype(int)
            df["day"] = df["day"].astype(int)
            df["period"] = df["period"].astype(int)
            df["price_spain"] = pd.to_numeric(df["price_spain"], errors="coerce")

            # Crear fecha
            df["date"] = pd.to_datetime(df[["year", "month", "day"]])

            # Detectar horario vs cuartohorario
            if df["period"].max() <= 24:
                df["datetime"] = df["date"] + pd.to_timedelta(df["period"] - 1, unit="h")
            else:
                df["datetime"] = df["date"] + pd.to_timedelta((df["period"] - 1) * 15, unit="m")

            all_data.append(df[["datetime", "price_spain"]])

        except Exception as e:
            print(f"Error en {file}: {e}")

# UNIR TODO

final_df = pd.concat(all_data, ignore_index=True)

# Ordenar
final_df = final_df.sort_values("datetime").reset_index(drop=True)

# Eliminar duplicados
final_df = final_df.drop_duplicates(subset="datetime")

# CONVERTIR A UTC (ROBUSTO)

final_df["datetime"] = pd.to_datetime(final_df["datetime"])

final_df["datetime"] = final_df["datetime"].dt.tz_localize(
    "Europe/Madrid",
    ambiguous="NaT",
    nonexistent="shift_forward"
)

# eliminar problemas de DST
final_df = final_df.dropna(subset=["datetime"])

# pasar a UTC
final_df["datetime"] = final_df["datetime"].dt.tz_convert("UTC")
final_df["datetime"] = final_df["datetime"].dt.tz_convert(None)


# HOMOGENEIZAR A HORARIO

df_precios = (
    final_df
    .set_index("datetime")
    .resample("h")
    .mean()
    .reset_index()
)


# GUARDADO INCREMENTAL

OMIE_PATH = "precio_omie_utc.csv"

if os.path.exists(OMIE_PATH):
    df_existing_omie = pd.read_csv(OMIE_PATH, parse_dates=["datetime"])
else:
    df_existing_omie = None

# asegurar tipos
df_precios["datetime"] = pd.to_datetime(df_precios["datetime"])

if df_existing_omie is not None:
    df_existing_omie["datetime"] = pd.to_datetime(df_existing_omie["datetime"])

# guardar
if df_existing_omie is None:
    
    df_precios = df_precios.sort_values("datetime")
    df_precios.to_csv(OMIE_PATH, index=False)
    
    print("✅ Dataset OMIE creado")

else:
    
    df_updated = pd.concat([df_existing_omie, df_precios])
    
    df_updated = df_updated.drop_duplicates(subset=["datetime"])
    df_updated = df_updated.sort_values("datetime")
    
    df_updated.to_csv(OMIE_PATH, index=False)
    
    print("✅ Dataset OMIE actualizado")

# INFO FINAL
print("Total registros:", len(df_precios))
print("Rango:", df_precios["datetime"].min(), "→", df_precios["datetime"].max())
print(df_precios.head())

✅ Dataset OMIE actualizado
Total registros: 72623
Rango: 2017-12-31 23:00:00 → 2026-04-14 21:00:00
             datetime  price_spain
0 2017-12-31 23:00:00         28.1
1 2018-01-01 00:00:00         33.0
2 2018-01-01 01:00:00         32.9
3 2018-01-01 02:00:00         28.1
4 2018-01-01 03:00:00         27.6


## REE

Mediante api publicca de REE se saca la demanda programada, la real y se calcula el error

In [5]:
# 1. FUNCIÓN API

def get_demanda(start, end):
    url = "https://apidatos.ree.es/es/datos/demanda/demanda-tiempo-real"
    
    params = {
        "start_date": start,
        "end_date": end,
        "time_trunc": "hour"
    }
    
    headers = {"Accept": "application/json"}
    
    try:
        response = requests.get(url, params=params, headers=headers)
        
        if response.status_code != 200:
            print(f"❌ Error {response.status_code}: {start} → {end}")
            return None
        
        return response.json()
    
    except:
        print(f"❌ Fallo conexión: {start} → {end}")
        return None



# 2. DESCARGA POR BLOQUES 

def descargar_demanda(start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    current = start
    all_data = []
    
    while current <= end:
        next_date = min(current + timedelta(days=7), end)
        
        print(f"⬇️ {current.date()} → {next_date.date()}")
        
        data = get_demanda(
            current.strftime("%Y-%m-%dT00:00"),
            next_date.strftime("%Y-%m-%dT23:59")
        )
        
        if data is not None:
            all_data.append(data)
        
        current = next_date + timedelta(days=1)  # evitar solapamiento
    
    return all_data


# 3. JSON → DATAFRAME

def transformar_json(all_data):
    data_list = []
    
    for data in all_data:
        if data is None or "included" not in data:
            continue
        
        for item in data["included"]:
            nombre = item["attributes"]["title"]
            
            for v in item["attributes"]["values"]:
                data_list.append({
                    "datetime": v["datetime"],
                    "variable": nombre,
                    "valor": v["value"]
                })
    
    df = pd.DataFrame(data_list)
    
    return df


# 4. LIMPIEZA Y FORMATO FINAL

def preparar_dataset(df):
    df["datetime"] = pd.to_datetime(df["datetime"], utc= True)
    
    df_pivot = df.pivot_table(
        index="datetime",
        columns="variable",
        values="valor",
        aggfunc="mean"
    )
    
    df_pivot = df_pivot.sort_index()
    
    # Variables útiles
    if "Real" in df_pivot.columns and "Prevista" in df_pivot.columns:
        df_pivot["error_demanda"] = df_pivot["Real"] - df_pivot["Prevista"]
    
    return df_pivot


# 5. EJECUCIÓN FINAL

all_data = descargar_demanda(
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
)

df_raw = transformar_json(all_data)

df_demanda = preparar_dataset(df_raw)

print("\n✅ DATASET FINAL DEMANDA:")
print(df_demanda.head())

print("\n📊 INFO:")
print(df_demanda.info())

⬇️ 2026-02-08 → 2026-02-15
⬇️ 2026-02-16 → 2026-02-23
⬇️ 2026-02-24 → 2026-03-03
⬇️ 2026-03-04 → 2026-03-11
⬇️ 2026-03-12 → 2026-03-19
⬇️ 2026-03-20 → 2026-03-27
⬇️ 2026-03-28 → 2026-04-04
⬇️ 2026-04-05 → 2026-04-12
⬇️ 2026-04-13 → 2026-04-14

✅ DATASET FINAL DEMANDA:
variable                   Prevista  Programada  Programada total     Real  \
datetime                                                                     
2026-02-07 23:00:00+00:00   27349.0     27111.0           27061.0  27662.0   
2026-02-07 23:05:00+00:00   27134.0     27111.0           27061.0  27352.0   
2026-02-07 23:10:00+00:00   26933.0     27111.0           27061.0  27276.0   
2026-02-07 23:15:00+00:00   26747.0     26611.0           26561.0  26982.0   
2026-02-07 23:20:00+00:00   26567.0     26611.0           26561.0  26745.0   

variable                   error_demanda  
datetime                                  
2026-02-07 23:00:00+00:00          313.0  
2026-02-07 23:05:00+00:00          218.0  
2026-02-07 2

In [6]:
df_demanda = (
    df_demanda
    .resample("h")
    .mean()
)
df_demanda = df_demanda.reset_index()

In [7]:
DEMANDA_PATH = "demanda_horaria.csv"

if os.path.exists(DEMANDA_PATH):
    df_existing_demanda = pd.read_csv(DEMANDA_PATH, parse_dates=["datetime"])
else:
    df_existing_demanda = None

# asegurar tipos
df_demanda["datetime"] = pd.to_datetime(df_demanda["datetime"])

if df_existing_demanda is not None:
    df_existing_demanda["datetime"] = pd.to_datetime(df_existing_demanda["datetime"])

# GUARDADO

if df_existing_demanda is None:
    
    df_demanda = df_demanda.sort_values("datetime")
    df_demanda.to_csv(DEMANDA_PATH, index=False)
    
    print("✅ Dataset demanda creado")

else:
    
    df_updated = pd.concat([df_existing_demanda, df_demanda])
    
    df_updated = df_updated.drop_duplicates(subset=["datetime"])
    df_updated = df_updated.sort_values("datetime")
    
    df_updated.to_csv(DEMANDA_PATH, index=False)
    
    print("✅ Dataset demanda actualizado")

✅ Dataset demanda actualizado


Aqui se combinan ambos precio y demanda para crear un dataset base

In [8]:
print(df_precios.head())
print(df_demanda.head())

print(df_precios.dtypes)
print(df_demanda.dtypes)

df_precios["datetime"] = pd.to_datetime(df_precios["datetime"]).dt.tz_localize(None)
df_demanda["datetime"] = pd.to_datetime(df_demanda["datetime"]).dt.tz_localize(None)

df_model = pd.merge(df_precios, df_demanda, on="datetime", how="left")

             datetime  price_spain
0 2017-12-31 23:00:00         28.1
1 2018-01-01 00:00:00         33.0
2 2018-01-01 01:00:00         32.9
3 2018-01-01 02:00:00         28.1
4 2018-01-01 03:00:00         27.6
variable                  datetime      Prevista  Programada  \
0        2026-02-07 23:00:00+00:00  26309.916667    26409.00   
1        2026-02-08 00:00:00+00:00  24340.000000    24490.75   
2        2026-02-08 01:00:00+00:00  23016.000000    23315.75   
3        2026-02-08 02:00:00+00:00  22218.083333    22476.25   
4        2026-02-08 03:00:00+00:00  22133.500000    22177.50   

variable  Programada total          Real  error_demanda  
0                 26359.00  26685.500000     375.583333  
1                 24451.75  24901.000000     561.000000  
2                 23279.75  23508.250000     492.250000  
3                 22443.25  22629.250000     411.166667  
4                 22145.50  22277.416667     143.916667  
datetime       datetime64[ns]
price_spain           float

In [9]:
print(df_model)

                 datetime  price_spain      Prevista  Programada  \
0     2017-12-31 23:00:00      28.1000           NaN         NaN   
1     2018-01-01 00:00:00      33.0000           NaN         NaN   
2     2018-01-01 01:00:00      32.9000           NaN         NaN   
3     2018-01-01 02:00:00      28.1000           NaN         NaN   
4     2018-01-01 03:00:00      27.6000           NaN         NaN   
...                   ...          ...           ...         ...   
72618 2026-04-14 17:00:00      54.5200  32022.583333    30425.50   
72619 2026-04-14 18:00:00      97.1150  33268.583333    33023.00   
72620 2026-04-14 19:00:00     114.8050  34342.250000    34289.00   
72621 2026-04-14 20:00:00      98.8675  31546.083333    31193.00   
72622 2026-04-14 21:00:00      92.8750  28256.833333    27994.75   

       Programada total          Real  error_demanda  
0                   NaN           NaN            NaN  
1                   NaN           NaN            NaN  
2                 

In [10]:
PD_PATH = "dataset_pd.csv"

# CARGAR SI EXISTE

if os.path.exists(PD_PATH):
    df_existing_pd = pd.read_csv(PD_PATH, parse_dates=["datetime"])
else:
    df_existing_pd = None

# asegurar tipos
df_model["datetime"] = pd.to_datetime(df_model["datetime"])

if df_existing_pd is not None:
    df_existing_pd["datetime"] = pd.to_datetime(df_existing_pd["datetime"])

# GUARDADO

if df_existing_pd is None:
    
    df_model = df_model.sort_values("datetime")
    df_model.to_csv(PD_PATH, index=False)
    
    print("✅ Dataset PD creado")

else:
    
    df_updated = pd.concat([df_existing_pd, df_model])
    
    df_updated = df_updated.drop_duplicates(subset=["datetime"])
    df_updated = df_updated.sort_values("datetime")
    
    df_updated.to_csv(PD_PATH, index=False)
    
    print("✅ Dataset PD actualizado")

✅ Dataset PD actualizado


## Open-Meteo

Se usa la api publica de open meteo para sacar datos de la temperatura y velocidad del viento. importante para generacion eolica y fotovoltaica

In [11]:
def get_weather_block(start_date, end_date):
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": 40.4168,
        "longitude": -3.7038,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,wind_speed_10m",
        "timezone": "UTC"
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"❌ Error clima: {start_date} → {end_date}")
        return None
    
    data = response.json()
    
    df = pd.DataFrame({
        "datetime": data["hourly"]["time"],
        "temperature": data["hourly"]["temperature_2m"],
        "wind_speed": data["hourly"]["wind_speed_10m"],
    })
    
    df["datetime"] = pd.to_datetime(df["datetime"])
    
    return df


def descargar_clima(start_date, end_date):
    
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    current = start
    all_data = []
    
    while current < end:
        next_date = min(current + timedelta(days=30), end)
        
        print(f"⬇️ Clima: {current.date()} → {next_date.date()}")
        
        df = get_weather_block(
            current.strftime("%Y-%m-%d"),
            next_date.strftime("%Y-%m-%d")
        )
        
        if df is not None:
            all_data.append(df)
        
        current = next_date + timedelta(days=1)
    
    return pd.concat(all_data, ignore_index=True)


# EJECUCIÓN
df_clima = descargar_clima(
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
)

print(df_clima.head())
print(df_clima.shape)

⬇️ Clima: 2026-02-08 → 2026-03-10
⬇️ Clima: 2026-03-11 → 2026-04-10
⬇️ Clima: 2026-04-11 → 2026-04-14
             datetime  temperature  wind_speed
0 2026-02-08 00:00:00          6.1        23.1
1 2026-02-08 01:00:00          5.8        25.1
2 2026-02-08 02:00:00          5.4        22.5
3 2026-02-08 03:00:00          5.1        18.8
4 2026-02-08 04:00:00          5.1        20.4
(1584, 3)


In [12]:
df_model = df_model.merge(df_clima, on="datetime", how="left")

In [13]:
PDTWS_PATH = "dataset_pdtws.csv"

# CARGAR SI EXISTE

if os.path.exists(PDTWS_PATH):
    df_existing_pdtws = pd.read_csv(PDTWS_PATH, parse_dates=["datetime"])
else:
    df_existing_pdtws = None

# asegurar tipos
df_model["datetime"] = pd.to_datetime(df_model["datetime"])

if df_existing_pdtws is not None:
    df_existing_pdtws["datetime"] = pd.to_datetime(df_existing_pdtws["datetime"])

# GUARDADO

if df_existing_pdtws is None:
    
    df_model = df_model.sort_values("datetime")
    df_model.to_csv(PDTWS_PATH, index=False)
    
    print("✅ Dataset PDTWS creado")

else:
    
    df_updated = pd.concat([df_existing_pdtws, df_model])
    
    df_updated = df_updated.drop_duplicates(subset=["datetime"])
    df_updated = df_updated.sort_values("datetime")
    
    df_updated.to_csv(PDTWS_PATH, index=False)
    
    print("✅ Dataset PDTWS actualizado")

✅ Dataset PDTWS actualizado


In [14]:
df_model["hour"] = df_model["datetime"].dt.hour
df_model["dayofweek"] = df_model["datetime"].dt.dayofweek
df_model["month"] = df_model["datetime"].dt.month

## Variables ciclicas

se calculan algunas variables ciclicas para que el modelo aprenda que es un ciclo.

ej 12 --> 0 el modelo piensa que no estan cerca

estas variables aseguran que el modelo sepa que  es un ciclo

```
  0
 / \
12  1
```

In [15]:
df_model["hour_sin"] = np.sin(2 * np.pi * df_model["hour"] / 24)
df_model["hour_cos"] = np.cos(2 * np.pi * df_model["hour"] / 24)

df_model["day_sin"] = np.sin(2 * np.pi * df_model["dayofweek"] / 7)
df_model["day_cos"] = np.cos(2 * np.pi * df_model["dayofweek"] / 7)

df_model["month_sin"] = np.sin(2 * np.pi * df_model["month"] / 12)
df_model["month_cos"] = np.cos(2 * np.pi * df_model["month"] / 12)

## Variables autoregressivas

se añaden el precio de la hora anteriror y 24 horas antes para añadir contexto

In [16]:
# --- FIX TIMEZONE ---
df_model["datetime"] = pd.to_datetime(df_model["datetime"], utc=True)

if df_existing is not None:
    df_existing["datetime"] = pd.to_datetime(df_existing["datetime"], utc=True)

# --- CONCAT PARA LAGS ---
if df_existing is not None:
    df_model = pd.concat([df_existing.tail(200), df_model])

# --- ORDEN ---
df_model = df_model.sort_values("datetime")

# --- LAGS ---
df_model["price_lag_1"] = df_model["price_spain"].shift(1)
df_model["price_lag_24"] = df_model["price_spain"].shift(24)

# --- FILTRAR SOLO NUEVO ---
if df_existing is not None:
    cutoff = df_existing["datetime"].max()
    df_model = df_model[df_model["datetime"] > cutoff]

In [17]:
df_model

,datetime,hour,dayofweek,month,price_spain,Prevista,Programada,Real,error_demanda,temperature,...,day_cos,month_sin,month_cos,price_lag_1,price_lag_24,is_weekend,is_holiday,gas_price,co2_price,Programada total
72144,2026-03-25 23:00:00+00:00,23,2,3,5.3000,26539.166667,25811.75,26041.416667,-497.750000,10.6,...,-0.222521,1.000000,6.123234e-17,20.0400,-0.8300,NaN,NaN,NaN,NaN,25771.75
72145,2026-03-26 00:00:00+00:00,0,3,3,3.5775,24927.333333,24299.00,24516.833333,-410.500000,9.6,...,-0.900969,1.000000,6.123234e-17,5.3000,-0.8300,NaN,NaN,NaN,NaN,24264.00
72146,2026-03-26 01:00:00+00:00,1,3,3,3.5100,23845.666667,23264.25,23576.833333,-268.833333,8.4,...,-0.900969,1.000000,6.123234e-17,3.5775,-0.8750,NaN,NaN,NaN,NaN,23232.25
72147,2026-03-26 02:00:00+00:00,2,3,3,3.5100,23373.833333,22921.75,23088.833333,-285.000000,7.6,...,-0.900969,1.000000,6.123234e-17,3.5100,-0.8750,NaN,NaN,NaN,NaN,22890.75
72148,2026-03-26 03:00:00+00:00,3,3,3,3.5100,23160.666667,22979.25,23117.750000,-42.916667,6.8,...,-0.900969,1.000000,6.123234e-17,3.5100,-0.8725,NaN,NaN,NaN,NaN,22949.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72618,2026-04-14 17:00:00+00:00,17,1,4,54.5200,32022.583333,30425.50,32422.000000,399.416667,19.9,...,0.623490,0.866025,-5.000000e-01,3.7950,26.9575,NaN,NaN,NaN,NaN,32130.75
72619,2026-04-14 18:00:00+00:00,18,1,4,97.1150,33268.583333,33023.00,33401.666667,133.083333,19.6,...,0.623490,0.866025,-5.000000e-01,54.5200,82.2350,NaN,NaN,NaN,NaN,33317.50
72620,2026-04-14 19:00:00+00:00,19,1,4,114.8050,34342.250000,34289.00,34029.583333,-312.666667,18.1,...,0.623490,0.866025,-5.000000e-01,97.1150,99.9500,NaN,NaN,NaN,NaN,34231.25
72621,2026-04-14 20:00:00+00:00,20,1,4,98.8675,31546.083333,31193.00,31363.083333,-183.000000,17.0,...,0.623490,0.866025,-5.000000e-01,114.8050,89.7750,NaN,NaN,NaN,NaN,31138.00


# ESIOS GENERACION POR TECNOLOGIA

Se saca mediante api de esos los datos de generacion de energia eolica y fotovoltaica horarias desde 2020 hasta ahora

In [18]:
API_KEY = "d28ad265f7d2b17ae5bf3c54d39e1dcc915d822a01e3c0f28449fadd089cff14"

In [19]:
def get_esios(indicator, start, end):
    
    url = f"https://api.esios.ree.es/indicators/{indicator}"
    
    headers = {
        "Accept": "application/json; application/vnd.esios-api-v2+json",
        "Content-Type": "application/json",
        "x-api-key": API_KEY
    }
    
    params = {
        "start_date": start,
        "end_date": end,
        "time_trunc": "hour"
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code != 200:
        print("❌ Error ESIOS:", response.status_code)
        return None
    
    data = response.json()
    
    values = data.get("indicator", {}).get("values", [])
    
    if not values:
        print("⚠️ Sin datos")
        return None
    
    df = pd.DataFrame(values)
    
    # detectar columna de tiempo automáticamente
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    elif "datetime_utc" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime_utc"], utc=True)
    else:
        print("❌ No hay columna datetime")
        print(df.columns)
        return None
    
    # quitar timezone
    df["datetime"] = df["datetime"].dt.tz_convert(None)
    
    return df[["datetime", "value"]]

In [20]:
# Eólica
WIND_ID = 10037

# Solar fotovoltaica
SOLAR_ID = 1161

# Solar termica 
SOLAR_T_ID = 1162 

# Biogas 
BIOGAS_ID = 1169 

# Carbon 
CARBON_ID = 10036 

# Ciclo Combinado 
CICLO_COMBINADO_ID = 1156 

# Cogeneracion 
COGENERACION_ID = 10039 

# Hidráulica
HIDRAULICA_ID = 10033  

# Consumo Bombeo 
CONSUMO_BOMBEO_ID = 1172 

# Nuclear 
NUCLEAR_ID = 1153 

# Turbinacion bombeo 
TURBINACION_BOMBEO_ID = 1152

# Importacion/Exportacion Francia
IMP_FRA_ID = 1174
EXP_FRA_ID = 1178

# Importacion/Exportacion Portugal
IMP_POR_ID = 1175
EXP_POR_ID = 1179

# Importacion/Exportacion Marruecos
IMP_MAR_ID =1176
EXP_MAR_ID = 1180

# Importacion/Exportacion Andorra
IMP_AND_ID = 1181
EXP_AND_ID = 1177

In [21]:
def descargar_esios(indicator, start_date, end_date):
    
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    current = start
    all_data = []
    
    while current < end:
        next_date = min(current + timedelta(days=15), end)
        
        print(f"⬇️ {indicator}: {current.date()} → {next_date.date()}")
        
        df = get_esios(
            indicator,
            current.strftime("%Y-%m-%dT00:00"),
            next_date.strftime("%Y-%m-%dT23:59")
        )
        
        if df is not None and not df.empty:
            all_data.append(df)
        else:
            print("⚠️ Sin datos")
        
        current = next_date + timedelta(days=1)
    
    # CASO: NO HAY DATOS
    
    if len(all_data) == 0:
        print("⚠️ No hay datos en todo el rango")
        
        date_range = pd.date_range(start=start, end=end, freq="h")
        
        return pd.DataFrame({
            "datetime": date_range,
            "value": [None] * len(date_range)
        })
    
  
    # CASO: HAY DATOS

    df = pd.concat(all_data, ignore_index=True)

    # asegurar datetime
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert(None)

    # ELIMINAR DUPLICADOS PRIMERO
    df = df.groupby("datetime", as_index=False).mean()

    # CREAR RANGO COMPLETO DESPUÉS
    full_range = pd.date_range(
        start=df["datetime"].min(),
        end=df["datetime"].max(),
        freq="h"
    )

    df = df.set_index("datetime").reindex(full_range)

    # 🔥 RELLENAR
    if df["value"].notna().sum() > 0:
        df["value"] = df["value"].ffill()

    df = df.reset_index().rename(columns={"index": "datetime"})

    return df

DESCARGA INTERCONEXIONES

In [22]:
df_imp_fra = descargar_esios(
    IMP_FRA_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "import_francia"})

df_exp_fra = descargar_esios(
    EXP_FRA_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "export_francia"})

df_imp_por = descargar_esios(
    IMP_POR_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "import_portugal"})

df_exp_por = descargar_esios(
    EXP_POR_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "export_portugal"})

df_imp_mar = descargar_esios(
    IMP_MAR_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "import_marruecos"})

df_exp_mar = descargar_esios(
    EXP_MAR_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "export_marruecos"})

df_imp_and = descargar_esios(
    IMP_AND_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "import_andorra"})

df_exp_and = descargar_esios(
    EXP_AND_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "export_andorra"})

⬇️ 1174: 2026-02-08 → 2026-02-23
⬇️ 1174: 2026-02-24 → 2026-03-11
⬇️ 1174: 2026-03-12 → 2026-03-27
⬇️ 1174: 2026-03-28 → 2026-04-12
⬇️ 1174: 2026-04-13 → 2026-04-14
⚠️ Sin datos
⚠️ Sin datos
⬇️ 1178: 2026-02-08 → 2026-02-23
⬇️ 1178: 2026-02-24 → 2026-03-11
⬇️ 1178: 2026-03-12 → 2026-03-27
⬇️ 1178: 2026-03-28 → 2026-04-12
⬇️ 1178: 2026-04-13 → 2026-04-14
⚠️ Sin datos
⚠️ Sin datos
⬇️ 1175: 2026-02-08 → 2026-02-23
⬇️ 1175: 2026-02-24 → 2026-03-11
⬇️ 1175: 2026-03-12 → 2026-03-27
⬇️ 1175: 2026-03-28 → 2026-04-12
⬇️ 1175: 2026-04-13 → 2026-04-14
⚠️ Sin datos
⚠️ Sin datos
⬇️ 1179: 2026-02-08 → 2026-02-23
⬇️ 1179: 2026-02-24 → 2026-03-11
⬇️ 1179: 2026-03-12 → 2026-03-27
⬇️ 1179: 2026-03-28 → 2026-04-12
⬇️ 1179: 2026-04-13 → 2026-04-14
⚠️ Sin datos
⚠️ Sin datos
⬇️ 1176: 2026-02-08 → 2026-02-23
⬇️ 1176: 2026-02-24 → 2026-03-11
⬇️ 1176: 2026-03-12 → 2026-03-27
⬇️ 1176: 2026-03-28 → 2026-04-12
⬇️ 1176: 2026-04-13 → 2026-04-14
⚠️ Sin datos
⚠️ Sin datos
⬇️ 1180: 2026-02-08 → 2026-02-23
⬇️ 1180: 202

UTC + AGREGACION

In [23]:
def limpiar_y_agregar(df, col_name):
    df["datetime"] = (
        pd.to_datetime(df["datetime"], utc=True)
        .dt.tz_convert("UTC")
        .dt.tz_localize(None)
    )
    
    df = (
        df.groupby("datetime", as_index=False)
        .agg({col_name: "sum"})
    )
    
    return df

APLICAR LIMPIEZA

In [24]:
df_imp_fra = limpiar_y_agregar(df_imp_fra, "import_francia")
df_exp_fra = limpiar_y_agregar(df_exp_fra, "export_francia")

df_imp_por = limpiar_y_agregar(df_imp_por, "import_portugal")
df_exp_por = limpiar_y_agregar(df_exp_por, "export_portugal")

df_imp_mar = limpiar_y_agregar(df_imp_mar, "import_marruecos")
df_exp_mar = limpiar_y_agregar(df_exp_mar, "export_marruecos")

df_imp_and = limpiar_y_agregar(df_imp_and, "import_andorra")
df_exp_and = limpiar_y_agregar(df_exp_and, "export_andorra")

MERGE POR PAÍS + NETO

In [25]:
def merge_pais(df_imp, df_exp, nombre):
    df = df_imp.merge(df_exp, on="datetime", how="outer").fillna(0)
    
    # neto = import + export (export ya viene negativo)
    df[f"net_{nombre}"] = df[df_imp.columns[1]] + df[df_exp.columns[1]]
    
    return df[["datetime", f"net_{nombre}"]]

In [26]:
df_fra = merge_pais(df_imp_fra, df_exp_fra, "francia")
df_por = merge_pais(df_imp_por, df_exp_por, "portugal")
df_mar = merge_pais(df_imp_mar, df_exp_mar, "marruecos")
df_and = merge_pais(df_imp_and, df_exp_and, "andorra")

NET IMPORTS TOTAL

In [27]:
df_inter = df_fra.merge(df_por, on="datetime", how="outer")
df_inter = df_inter.merge(df_mar, on="datetime", how="outer")
df_inter = df_inter.merge(df_and, on="datetime", how="outer")

df_inter = df_inter.fillna(0)

df_inter["net_imports"] = (
    df_inter["net_francia"]
    + df_inter["net_portugal"]
    + df_inter["net_marruecos"]
    + df_inter["net_andorra"]
)

df_inter = df_inter[["datetime", "net_imports"]]

GENERACIÓN

In [28]:
df_wind = descargar_esios(
    WIND_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "wind_generation"})

df_solar = descargar_esios(
    SOLAR_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "solar_generation"})

df_ciclo = descargar_esios(
    CICLO_COMBINADO_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "ciclo_combinado_generation"})

df_hidraulica = descargar_esios(
    HIDRAULICA_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "hidraulica_generation"})

df_consumo_bombeo = descargar_esios(
    CONSUMO_BOMBEO_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "consumo_bombeo"})

df_turbinacion = descargar_esios(
    TURBINACION_BOMBEO_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "turbinacion_bombeo"})

df_nuclear = descargar_esios(
    NUCLEAR_ID,
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
).rename(columns={"value": "nuclear_generation"})

df_wind = limpiar_y_agregar(df_wind, "wind_generation")
df_solar = limpiar_y_agregar(df_solar, "solar_generation")
df_ciclo = limpiar_y_agregar(df_ciclo, "ciclo_combinado_generation")
df_hidraulica = limpiar_y_agregar(df_hidraulica, "hidraulica_generation")
df_consumo_bombeo = limpiar_y_agregar(df_consumo_bombeo, "consumo_bombeo")
df_turbinacion = limpiar_y_agregar(df_turbinacion, "turbinacion_bombeo")
df_nuclear = limpiar_y_agregar(df_nuclear, "nuclear_generation")

df_gen = df_wind.merge(df_solar, on="datetime", how="outer")
df_gen = df_gen.merge(df_ciclo, on="datetime", how="outer")
df_gen = df_gen.merge(df_hidraulica, on="datetime", how="outer")
df_gen = df_gen.merge(df_consumo_bombeo, on="datetime", how="outer")
df_gen = df_gen.merge(df_turbinacion, on="datetime", how="outer")
df_gen = df_gen.merge(df_nuclear, on="datetime", how="outer")

⬇️ 10037: 2026-02-08 → 2026-02-23
⬇️ 10037: 2026-02-24 → 2026-03-11
⬇️ 10037: 2026-03-12 → 2026-03-27
⬇️ 10037: 2026-03-28 → 2026-04-12
⬇️ 10037: 2026-04-13 → 2026-04-14
⚠️ Sin datos
⚠️ Sin datos
⬇️ 1161: 2026-02-08 → 2026-02-23
⬇️ 1161: 2026-02-24 → 2026-03-11
⬇️ 1161: 2026-03-12 → 2026-03-27
⬇️ 1161: 2026-03-28 → 2026-04-12
⬇️ 1161: 2026-04-13 → 2026-04-14
⚠️ Sin datos
⚠️ Sin datos
⬇️ 1156: 2026-02-08 → 2026-02-23
⬇️ 1156: 2026-02-24 → 2026-03-11
⬇️ 1156: 2026-03-12 → 2026-03-27
⬇️ 1156: 2026-03-28 → 2026-04-12
⬇️ 1156: 2026-04-13 → 2026-04-14
⚠️ Sin datos
⚠️ Sin datos
⬇️ 10033: 2026-02-08 → 2026-02-23
⬇️ 10033: 2026-02-24 → 2026-03-11
⬇️ 10033: 2026-03-12 → 2026-03-27
⬇️ 10033: 2026-03-28 → 2026-04-12
⬇️ 10033: 2026-04-13 → 2026-04-14
⬇️ 1172: 2026-02-08 → 2026-02-23
⬇️ 1172: 2026-02-24 → 2026-03-11
⬇️ 1172: 2026-03-12 → 2026-03-27
⬇️ 1172: 2026-03-28 → 2026-04-12
⬇️ 1172: 2026-04-13 → 2026-04-14
⚠️ Sin datos
⚠️ Sin datos
⬇️ 1152: 2026-02-08 → 2026-02-23
⬇️ 1152: 2026-02-24 → 2026-0

MERGE FINAL

In [29]:
df_final = df_gen.merge(df_inter, on="datetime", how="outer")

print(df_final.head())
print(df_final.shape)

             datetime  wind_generation  solar_generation  \
0 2026-02-07 23:00:00       211.172095          0.006400   
1 2026-02-08 00:00:00       180.015488          0.013261   
2 2026-02-08 01:00:00       162.515103          0.006500   
3 2026-02-08 02:00:00       151.584462          0.006682   
4 2026-02-08 03:00:00       143.977333          0.008333   

   ciclo_combinado_generation  hidraulica_generation  consumo_bombeo  \
0                  210.763000            1048.283292     -155.103000   
1                  219.899818            1030.438733     -180.474182   
2                  219.816545            1022.359823     -182.404636   
3                  212.370364            1021.291970     -189.809545   
4                  209.106727            1020.951972     -191.890455   

   turbinacion_bombeo  nuclear_generation  net_imports  
0           69.200833          1591.25050    -197.6628  
1          108.395250          1592.12250      -7.4656  
2          107.940500          1591

GUARDAR

In [30]:
GEN_INTER_PATH = "dataset_generacion_interconexiones.csv"

# CARGAR SI EXISTE

if os.path.exists(GEN_INTER_PATH):
    df_existing_gen = pd.read_csv(GEN_INTER_PATH, parse_dates=["datetime"])
else:
    df_existing_gen = None

# asegurar tipos
df_final["datetime"] = pd.to_datetime(df_final["datetime"])

if df_existing_gen is not None:
    df_existing_gen["datetime"] = pd.to_datetime(df_existing_gen["datetime"])

# GUARDADO

if df_existing_gen is None:
    
    df_final = df_final.sort_values("datetime")
    df_final.to_csv(GEN_INTER_PATH, index=False)
    
    print("✅ Dataset gen+inter creado")

else:
    
    df_updated = pd.concat([df_existing_gen, df_final])
    
    df_updated = df_updated.drop_duplicates(subset=["datetime"])
    df_updated = df_updated.sort_values("datetime")
    
    df_updated.to_csv(GEN_INTER_PATH, index=False)
    
    print("✅ Dataset gen+inter actualizado")

✅ Dataset gen+inter actualizado


### Festivos/Finde Semana

In [31]:
# convertir a hora local España
df_model["datetime_local"] = (
    pd.to_datetime(df_model["datetime"], utc=True)
    .dt.tz_convert("Europe/Madrid")
)

# weekend
df_model["is_weekend"] = df_model["datetime_local"].dt.dayofweek.isin([5,6]).astype(int)

# holidays (FIX robusto)
spain_holidays = holidays.Spain()

df_model["is_holiday"] = df_model["datetime_local"].dt.date.apply(
    lambda x: x in spain_holidays
).astype(int)

# limpiar
df_model = df_model.drop(columns=["datetime_local"])

In [32]:
print(df_model["is_weekend"].sum())
print(df_model["is_holiday"].sum())

143
24


In [35]:
df_final["datetime"] = (
    pd.to_datetime(df_final["datetime"])
    .dt.tz_localize("UTC")
)

df_model = pd.merge(df_model, df_final, on="datetime", how="left")

### Feature Engeneering

In [37]:
df_model

,datetime,hour,dayofweek,month,price_spain,Prevista,Programada,Real,error_demanda,temperature,...,co2_price,Programada total,wind_generation_y,solar_generation_y,ciclo_combinado_generation_y,hidraulica_generation_y,consumo_bombeo_y,turbinacion_bombeo_y,nuclear_generation_y,net_imports_y
0,2026-03-25 23:00:00+00:00,23,2,3,5.3000,26539.166667,25811.75,26041.416667,-497.750000,10.6,...,NaN,25771.75,332.265732,0.004429,240.892250,1042.893431,-27.308000,47.621167,1530.84275,-2320.7202
1,2026-03-26 00:00:00+00:00,0,3,3,3.5775,24927.333333,24299.00,24516.833333,-410.500000,9.6,...,NaN,24264.00,321.402902,0.002222,210.158333,1051.781445,-59.434100,90.934833,1531.55150,-1750.9746
2,2026-03-26 01:00:00+00:00,1,3,3,3.5100,23845.666667,23264.25,23576.833333,-268.833333,8.4,...,NaN,23232.25,312.542244,0.005000,205.810917,1045.720571,-99.191800,105.295800,1531.12000,-1420.3032
3,2026-03-26 02:00:00+00:00,2,3,3,3.5100,23373.833333,22921.75,23088.833333,-285.000000,7.6,...,NaN,22890.75,314.406600,0.002667,199.294167,1047.383214,-105.963600,120.733000,1531.54350,-1366.7726
4,2026-03-26 03:00:00+00:00,3,3,3,3.5100,23160.666667,22979.25,23117.750000,-42.916667,6.8,...,NaN,22949.25,312.842050,0.003600,199.530083,1047.785981,-96.289273,102.981000,1531.81400,-1307.5726
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
474,2026-04-14 17:00:00+00:00,17,1,4,54.5200,32022.583333,30425.50,32422.000000,399.416667,19.9,...,NaN,32130.75,NaN,NaN,NaN,1035.157401,NaN,NaN,NaN,NaN
475,2026-04-14 18:00:00+00:00,18,1,4,97.1150,33268.583333,33023.00,33401.666667,133.083333,19.6,...,NaN,33317.50,NaN,NaN,NaN,948.468342,NaN,NaN,NaN,NaN
476,2026-04-14 19:00:00+00:00,19,1,4,114.8050,34342.250000,34289.00,34029.583333,-312.666667,18.1,...,NaN,34231.25,NaN,NaN,NaN,904.465207,NaN,NaN,NaN,NaN
477,2026-04-14 20:00:00+00:00,20,1,4,98.8675,31546.083333,31193.00,31363.083333,-183.000000,17.0,...,NaN,31138.00,NaN,NaN,NaN,924.058449,NaN,NaN,NaN,NaN


In [36]:
# generación total
df_model["generacion_total"] = (
    df_model["wind_generation"] +
    df_model["solar_generation"] +
    df_model["hidraulica_generation"] +
    df_model["ciclo_combinado_generation"]
)

# ratio renovable
df_model["renovable_ratio"] = (
    df_model["wind_generation"] +
    df_model["solar_generation"] +
    df_model["hidraulica_generation"]
) / df_model["generacion_total"]

# dependencia del gas
df_model["gas_ratio"] = (
    df_model["ciclo_combinado_generation"] / df_model["generacion_total"]
)

# bombeo neto
df_model["bombeo_neto"] = (
    df_model["turbinacion_bombeo"] - df_model["consumo_bombeo"]
)

# ratio nuclear
df_model["nuclear_ratio"] = (
    df_model["nuclear_generation"] / df_model["generacion_total"]
)

# base load
df_model["base_load"] = (
    df_model["nuclear_generation"] +
    df_model["hidraulica_generation"]
)

# base vs demanda ( cuanta de la demanda esta cubierta por generacion estable)
df_model["base_vs_demanda"] = (
    (df_model["nuclear_generation"] + df_model["hidraulica_generation"]) /
    (df_model["Prevista"] + 1e-6)
)

# gap de generacion
df_model["gap_no_renovable"] = (
    df_model["generacion_total"] -
    (df_model["wind_generation"] +
     df_model["solar_generation"] +
     df_model["nuclear_generation"])
)

KeyError: 'wind_generation'

#### TTF GAS

In [ ]:
# WEB --> https://es.investing.com/commodities/dutch-ttf-gas-c1-futures-historical-data

# 1. CARGAR CSV

df_gas = pd.read_csv("TTF_gas.csv")


# 2. PARSE FECHA (formato europeo)

df_gas["datetime"] = pd.to_datetime(df_gas["Fecha"], dayfirst=True)


# 3. LIMPIAR PRECIO

df_gas["gas_price"] = (
    df_gas["Último"]
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)


# 4. ORDENAR (IMPORTANTE)

df_gas = df_gas.sort_values("datetime")


# 5. CREAR FRECUENCIA DIARIA COMPLETA

df_gas = df_gas.set_index("datetime").asfreq("D")

# rellenar fines de semana
df_gas["gas_price"] = df_gas["gas_price"].ffill()


# 6. TIMEZONE → EUROPA → UTC

df_gas.index = df_gas.index.tz_localize("Europe/Madrid")
df_gas.index = df_gas.index.tz_convert("UTC")

# 7. PASAR A HORARIO

df_gas = df_gas.reset_index()

df_gas = (
    df_gas
    .set_index("datetime")
    .resample("h")
    .ffill()
    .reset_index()
)


# 8. QUEDARSE SOLO CON LO NECESARIO

df_gas = df_gas[["datetime", "gas_price"]]


# 9. FILTRAR RANGO

df_model["datetime"] = pd.to_datetime(df_model["datetime"], utc=True)

df_gas = df_gas[
    (df_gas["datetime"] >= df_model["datetime"].min()) &
    (df_gas["datetime"] <= df_model["datetime"].max())
]


# 10. MERGE FINAL 

# evitar duplicados si ya existe
if "gas_price" in df_model.columns:
    df_model = df_model.drop(columns=["gas_price"])

df_model = df_model.merge(
    df_gas,
    on="datetime",
    how="left"
)

# rellenar primer día (y posibles huecos)

df_model["gas_price"] = df_model["gas_price"].ffill().bfill()

# 11. CHECK FINAL

print("NaNs en gas:", df_model["gas_price"].isna().sum())
print("Columnas finales:", df_model.columns)

#### CO2 (EUA)

In [ ]:
# WEB --> https://www.investing.com/commodities/carbon-emissions-historical-data

# 1. CARGAR CSV

df_co2 = pd.read_csv("CO2.csv")


# 2. PARSE FECHA (formato europeo)

df_co2["datetime"] = pd.to_datetime(df_co2["Date"], dayfirst=True)


# 3. LIMPIAR PRECIO

df_co2["co2_price"] = (
    df_co2["Price"]
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)


# 4. ORDENAR

df_co2 = df_co2.sort_values("datetime")


# 5. FRECUENCIA DIARIA COMPLETA (rellenar findes)

df_co2 = df_co2.set_index("datetime").asfreq("D")


# rellenar fines de semana
df_co2["co2_price"] = df_co2["co2_price"].ffill()


# 6. TIMEZONE → EUROPA → UTC

df_co2.index = df_co2.index.tz_localize("Europe/Madrid")
df_co2.index = df_co2.index.tz_convert("UTC")


# 7. PASAR A HORARIO

df_co2 = df_co2.reset_index()

df_co2 = (
    df_co2
    .set_index("datetime")
    .resample("h")
    .ffill()
    .reset_index()
)


# 8. QUEDARSE SOLO CON LO NECESARIO

df_co2 = df_co2[["datetime", "co2_price"]]


# 9. FILTRAR RANGO

df_model["datetime"] = pd.to_datetime(df_model["datetime"], utc=True)

df_co2 = df_co2[
    (df_co2["datetime"] >= df_model["datetime"].min()) &
    (df_co2["datetime"] <= df_model["datetime"].max())
]


# 10. MERGE FINAL

if "co2_price" in df_model.columns:
    df_model = df_model.drop(columns=["co2_price"])

df_model = df_model.merge(
    df_co2,
    on="datetime",
    how="left"
)

# rellenar primer día (y posibles huecos)

df_model["co2_price"] = df_model["co2_price"].ffill().bfill()

# 11. CHECK FINAL

print("NaNs en CO2:", df_model["co2_price"].isna().sum())
print("Columnas finales:", df_model.columns)

In [ ]:
# CASO 1: primera ejecución

if df_existing is None:
    
    df_model = df_model.sort_values("datetime")
    df_model.to_csv(DATASET_FINAL_PATH, index=False)
    
    print("✅ Dataset creado completo")


# CASO 2: incremental

else:
    
    df_updated = pd.concat([df_existing, df_model])

    df_updated = df_updated.drop_duplicates(subset=["datetime"])
    df_updated = df_updated.sort_values("datetime")

    df_updated.to_csv(DATASET_FINAL_PATH, index=False)

    print("✅ Dataset actualizado")

---

# PARTE 2 — Variables previstas (ESIOS y Open-Meteo)

---

# Pipeline Variables Previstas

Este notebook añade al dataset_final_clean.csv las siguientes variables previstas:
- Generación programada PBF Eólica (ESIOS ID: 10073)
- Generación prevista Solar (ESIOS ID: 10034)
- Temperatura prevista (Open-Meteo Historical Forecast API)
- Velocidad del viento prevista (Open-Meteo Historical Forecast API)

Todo en UTC y frecuencia horaria.


In [11]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import time

# ── CONFIGURACION ─────────────────────────────────────────────────────────────
DATASET_PATH = 'dataset_final.csv'
API_KEY_ESIOS = 'd28ad265f7d2b17ae5bf3c54d39e1dcc915d822a01e3c0f28449fadd089cff14'

# Coordenadas Madrid (representativas del sistema peninsular)
LAT = 40.4168
LON = -3.7038

# Rango de fechas
START_DATE = '2020-01-01'
END_DATE = '2026-02-28'

print('Cargando dataset...')
df = pd.read_csv(DATASET_PATH)
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
df = df.sort_values('datetime').reset_index(drop=True)
print(f'Dataset cargado: {len(df)} registros')
print(f'Rango: {df["datetime"].min()} -> {df["datetime"].max()}')

Cargando dataset...
Dataset cargado: 54624 registros
Rango: 2019-12-31 23:00:00+00:00 -> 2026-03-25 22:00:00+00:00


## 1. Descarga de ESIOS


In [12]:
def get_esios(indicator_id, start, end, api_key, nombre=''):
    """
    Descarga datos de ESIOS por bloques mensuales para evitar timeouts.
    Devuelve DataFrame con columnas datetime (UTC, sin tz) y value.
    """
    print(f'Descargando {nombre} (ID: {indicator_id})...')
    
    headers = {
        'Accept': 'application/json; application/vnd.esios-api-v2+json',
        'Content-Type': 'application/json',
        'x-api-key': api_key
    }
    
    all_data = []
    
    current = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    
    while current <= end_ts:
        block_end = min(current + pd.DateOffset(months=1) - pd.Timedelta(hours=1), end_ts)
        
        params = {
            'start_date': current.strftime('%Y-%m-%dT%H:%M:%SZ'),
            'end_date': block_end.strftime('%Y-%m-%dT%H:%M:%SZ'),
            'time_trunc': 'hour'
        }
        
        url = f'https://api.esios.ree.es/indicators/{indicator_id}'
        response = requests.get(url, headers=headers, params=params)
        
        if response.status_code == 200:
            values = response.json().get('indicator', {}).get('values', [])
            if values:
                df_block = pd.DataFrame(values)
                if 'datetime' in df_block.columns:
                    df_block['datetime'] = pd.to_datetime(df_block['datetime'], utc=True)
                elif 'datetime_utc' in df_block.columns:
                    df_block['datetime'] = pd.to_datetime(df_block['datetime_utc'], utc=True)
                df_block['datetime'] = df_block['datetime'].dt.tz_convert(None)
                all_data.append(df_block[['datetime', 'value']])
                print(f'  {current.strftime("%Y-%m")} OK ({len(values)} registros)')
            else:
                print(f'  {current.strftime("%Y-%m")} sin datos')
        else:
            print(f'  {current.strftime("%Y-%m")} error {response.status_code}')
        
        current = current + pd.DateOffset(months=1)
        time.sleep(0.5)
    
    if not all_data:
        print(f'  Sin datos para {nombre}')
        return None
    
    result = pd.concat(all_data, ignore_index=True)
    result = result.drop_duplicates(subset='datetime')
    result = result.sort_values('datetime').reset_index(drop=True)
    print(f'  Total: {len(result)} registros')
    return result

In [13]:
# Descargar eolica prevista PBF
df_eolica_prev = get_esios(
    indicator_id=10073,
    start=START_DATE,
    end=END_DATE,
    api_key=API_KEY_ESIOS   ,
    nombre='Generacion programada PBF Eolica'
)

if df_eolica_prev is not None:
    df_eolica_prev = df_eolica_prev.rename(columns={'value': 'wind_generation_forecast'})
    print(df_eolica_prev.head())
    print(f'Rango: {df_eolica_prev["datetime"].min()} -> {df_eolica_prev["datetime"].max()}')

Descargando Generacion programada PBF Eolica (ID: 10073)...
  2020-01 OK (744 registros)
  2020-02 OK (696 registros)
  2020-03 OK (744 registros)
  2020-04 OK (720 registros)
  2020-05 OK (744 registros)
  2020-06 OK (720 registros)
  2020-07 OK (744 registros)
  2020-08 OK (744 registros)
  2020-09 OK (720 registros)
  2020-10 OK (744 registros)
  2020-11 OK (720 registros)
  2020-12 OK (744 registros)
  2021-01 OK (744 registros)
  2021-02 OK (672 registros)
  2021-03 OK (744 registros)
  2021-04 OK (720 registros)
  2021-05 OK (744 registros)
  2021-06 OK (720 registros)
  2021-07 OK (744 registros)
  2021-08 OK (744 registros)
  2021-09 OK (720 registros)
  2021-10 OK (744 registros)
  2021-11 OK (720 registros)
  2021-12 OK (744 registros)
  2022-01 OK (744 registros)
  2022-02 OK (672 registros)
  2022-03 OK (744 registros)
  2022-04 OK (720 registros)
  2022-05 OK (744 registros)
  2022-06 OK (720 registros)
  2022-07 OK (744 registros)
  2022-08 OK (744 registros)
  2022-09 OK

In [14]:
# Descargar solar prevista
df_solar_prev = get_esios(
    indicator_id=10034,
    start=START_DATE,
    end=END_DATE,
    api_key=API_KEY_ESIOS,
    nombre='Generacion prevista Solar'
)

if df_solar_prev is not None:
    df_solar_prev = df_solar_prev.rename(columns={'value': 'solar_generation_forecast'})
    print(df_solar_prev.head())
    print(f'Rango: {df_solar_prev["datetime"].min()} -> {df_solar_prev["datetime"].max()}')

Descargando Generacion prevista Solar (ID: 10034)...
  2020-01 OK (744 registros)
  2020-02 OK (696 registros)
  2020-03 OK (744 registros)
  2020-04 OK (720 registros)
  2020-05 OK (744 registros)
  2020-06 OK (720 registros)
  2020-07 OK (744 registros)
  2020-08 OK (744 registros)
  2020-09 OK (720 registros)
  2020-10 OK (744 registros)
  2020-11 OK (720 registros)
  2020-12 OK (744 registros)
  2021-01 OK (744 registros)
  2021-02 OK (672 registros)
  2021-03 OK (723 registros)
  2021-04 OK (720 registros)
  2021-05 OK (744 registros)
  2021-06 OK (720 registros)
  2021-07 OK (744 registros)
  2021-08 OK (744 registros)
  2021-09 OK (720 registros)
  2021-10 OK (744 registros)
  2021-11 OK (720 registros)
  2021-12 OK (744 registros)
  2022-01 OK (744 registros)
  2022-02 OK (672 registros)
  2022-03 OK (744 registros)
  2022-04 OK (720 registros)
  2022-05 OK (744 registros)
  2022-06 OK (720 registros)
  2022-07 OK (744 registros)
  2022-08 OK (744 registros)
  2022-09 OK (720 r

## 2. Descarga de Open-Meteo Historical Forecast API


In [15]:
def get_openmeteo_forecast(lat, lon, start, end):
    """
    Descarga forecasts historicos de Open-Meteo por bloques de 3 meses.
    Usa la Historical Forecast API que guarda los forecasts que se hicieron en el pasado.
    Devuelve DataFrame con datetime (UTC sin tz), temperature_forecast y wind_speed_forecast.
    """
    print('Descargando forecasts meteorologicos de Open-Meteo...')
    
    all_data = []
    
    current = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    
    while current <= end_ts:
        block_end = min(current + pd.DateOffset(months=3) - pd.Timedelta(days=1), end_ts)
        
        url = 'https://historical-forecast-api.open-meteo.com/v1/forecast'
        params = {
            'latitude': lat,
            'longitude': lon,
            'start_date': current.strftime('%Y-%m-%d'),
            'end_date': block_end.strftime('%Y-%m-%d'),
            'hourly': 'temperature_2m,wind_speed_10m',
            'timezone': 'UTC',
            'wind_speed_unit': 'ms'
        }
        
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            data = response.json()
            hourly = data.get('hourly', {})
            
            df_block = pd.DataFrame({
                'datetime': pd.to_datetime(hourly['time']),
                'temperature_forecast': hourly['temperature_2m'],
                'wind_speed_forecast': hourly['wind_speed_10m']
            })
            all_data.append(df_block)
            print(f'  {current.strftime("%Y-%m")} a {block_end.strftime("%Y-%m")} OK ({len(df_block)} registros)')
        else:
            print(f'  Error {response.status_code}: {response.text[:200]}')
        
        current = current + pd.DateOffset(months=3)
        time.sleep(1)
    
    if not all_data:
        print('Sin datos meteorologicos')
        return None
    
    result = pd.concat(all_data, ignore_index=True)
    result = result.drop_duplicates(subset='datetime')
    result = result.sort_values('datetime').reset_index(drop=True)
    print(f'Total: {len(result)} registros')
    return result

In [16]:
df_meteo_prev = get_openmeteo_forecast(
    lat=LAT,
    lon=LON,
    start=START_DATE,
    end=END_DATE
)

if df_meteo_prev is not None:
    print(df_meteo_prev.head())
    print(f'Rango: {df_meteo_prev["datetime"].min()} -> {df_meteo_prev["datetime"].max()}')

Descargando forecasts meteorologicos de Open-Meteo...
  2020-01 a 2020-03 OK (2184 registros)
  2020-04 a 2020-06 OK (2184 registros)
  2020-07 a 2020-09 OK (2208 registros)
  2020-10 a 2020-12 OK (2208 registros)
  2021-01 a 2021-03 OK (2160 registros)
  2021-04 a 2021-06 OK (2184 registros)
  2021-07 a 2021-09 OK (2208 registros)
  2021-10 a 2021-12 OK (2208 registros)
  2022-01 a 2022-03 OK (2160 registros)
  2022-04 a 2022-06 OK (2184 registros)
  2022-07 a 2022-09 OK (2208 registros)
  2022-10 a 2022-12 OK (2208 registros)
  2023-01 a 2023-03 OK (2160 registros)
  2023-04 a 2023-06 OK (2184 registros)
  2023-07 a 2023-09 OK (2208 registros)
  2023-10 a 2023-12 OK (2208 registros)
  2024-01 a 2024-03 OK (2184 registros)
  2024-04 a 2024-06 OK (2184 registros)
  2024-07 a 2024-09 OK (2208 registros)
  2024-10 a 2024-12 OK (2208 registros)
  2025-01 a 2025-03 OK (2160 registros)
  2025-04 a 2025-06 OK (2184 registros)
  2025-07 a 2025-09 OK (2208 registros)
  2025-10 a 2025-12 OK (22

C:\Users\Jaime Cremades\AppData\Local\Temp\ipykernel_12752\4063961845.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat(all_data, ignore_index=True)


## 3. Integrar al dataset


In [17]:
# Preparar datetime del dataset para el merge (sin timezone)
df['datetime_merge'] = df['datetime'].dt.tz_localize(None)

# Merge eolica prevista
if df_eolica_prev is not None:
    df_eolica_prev['datetime'] = pd.to_datetime(df_eolica_prev['datetime'])
    df = df.merge(df_eolica_prev, left_on='datetime_merge', right_on='datetime', how='left', suffixes=('', '_dup'))
    if 'datetime_dup' in df.columns:
        df = df.drop(columns=['datetime_dup'])
    print(f'Eolica prevista - nulos: {df["wind_generation_forecast"].isna().sum()}')

# Merge solar prevista
if df_solar_prev is not None:
    df_solar_prev['datetime'] = pd.to_datetime(df_solar_prev['datetime'])
    df = df.merge(df_solar_prev, left_on='datetime_merge', right_on='datetime', how='left', suffixes=('', '_dup'))
    if 'datetime_dup' in df.columns:
        df = df.drop(columns=['datetime_dup'])
    print(f'Solar prevista - nulos: {df["solar_generation_forecast"].isna().sum()}')

# Merge meteorologia prevista
if df_meteo_prev is not None:
    df_meteo_prev['datetime'] = pd.to_datetime(df_meteo_prev['datetime'])
    df = df.merge(df_meteo_prev, left_on='datetime_merge', right_on='datetime', how='left', suffixes=('', '_dup'))
    if 'datetime_dup' in df.columns:
        df = df.drop(columns=['datetime_dup'])
    print(f'Temperatura prevista - nulos: {df["temperature_forecast"].isna().sum()}')
    print(f'Viento previsto - nulos: {df["wind_speed_forecast"].isna().sum()}')

# Eliminar columna auxiliar
df = df.drop(columns=['datetime_merge'])

print(f'\nColumnas del dataset actualizado:')
print(list(df.columns))
print(f'Total registros: {len(df)}')

Eolica prevista - nulos: 659
Solar prevista - nulos: 645
Temperatura prevista - nulos: 11328
Viento previsto - nulos: 11328

Columnas del dataset actualizado:
['datetime', 'hour', 'dayofweek', 'month', 'price_spain', 'Prevista', 'Programada', 'Real', 'error_demanda', 'temperature', 'wind_speed', 'wind_generation', 'solar_generation', 'hidraulica_generation', 'nuclear_generation', 'ciclo_combinado_generation', 'generacion_total', 'net_imports', 'consumo_bombeo', 'turbinacion_bombeo', 'bombeo_neto', 'renovable_ratio', 'gas_ratio', 'nuclear_ratio', 'base_load', 'base_vs_demanda', 'gap_no_renovable', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'price_lag_1', 'price_lag_24', 'is_weekend', 'is_holiday', 'gas_price', 'co2_price', 'wind_generation_forecast', 'solar_generation_forecast', 'temperature_forecast', 'wind_speed_forecast']
Total registros: 54624


## 4. Imputacion de nulos y guardado


In [18]:
# Imputar nulos con forward fill y backward fill
nuevas_cols = ['wind_generation_forecast', 'solar_generation_forecast', 
               'temperature_forecast', 'wind_speed_forecast']

for col in nuevas_cols:
    if col in df.columns:
        antes = df[col].isna().sum()
        df[col] = df[col].ffill().bfill()
        despues = df[col].isna().sum()
        print(f'{col}: {antes} nulos -> {despues} nulos tras imputacion')

# Verificacion final
print('\nVerificacion final de nulos en nuevas columnas:')
print(df[nuevas_cols].isna().sum())

# Guardar
df.to_csv(DATASET_PATH, index=False)
print(f'\nDataset guardado en {DATASET_PATH}')
print(f'Total registros: {len(df)}')
print(f'Total columnas: {len(df.columns)}')

wind_generation_forecast: 659 nulos -> 0 nulos tras imputacion
solar_generation_forecast: 645 nulos -> 0 nulos tras imputacion
temperature_forecast: 11328 nulos -> 0 nulos tras imputacion
wind_speed_forecast: 11328 nulos -> 0 nulos tras imputacion

Verificacion final de nulos en nuevas columnas:
wind_generation_forecast     0
solar_generation_forecast    0
temperature_forecast         0
wind_speed_forecast          0
dtype: int64

Dataset guardado en dataset_final.csv
Total registros: 54624
Total columnas: 43


---

# PARTE 3 — Preprocesamiento final

---

# Preprocesamiento

muchas cosas ya se han hecho en el pipeline directo como los duplicados, cambiar a utc, los lags ...
aqui se acaba el preprocesamiento para hacer train/test split y hacer todos los modelos


In [6]:
import pandas as pd

### 0. Cargar el Dataset


In [7]:
df_model = pd.read_csv('dataset_final.csv')

### 1. LIMPIEZA + ORDEN

#### ASEGURAR DATETIME + UTC


In [8]:
df = df_model.copy()

df['datetime'] = pd.to_datetime(df['datetime'], utc=True)

In [9]:
# ordenar por tiempo
df = df.sort_values('datetime').reset_index(drop=True)

# eliminar duplicados
df = df.drop_duplicates(subset='datetime')

#### RECORTE TEMPORAL


In [10]:
cols_clave = [
    'price_spain',
    'gas_price',
    'Prevista', 'Programada', 'Real',
    'temperature', 'wind_speed',
    'wind_generation', 'solar_generation',
    'wind_generation_forecast',
    'solar_generation_forecast',
    'temperature_forecast',
    'wind_speed_forecast'
]

last_valid = df.dropna(subset=cols_clave)['datetime'].max()

df = df[df['datetime'] <= last_valid]

print('Fecha limite:', last_valid)

Fecha limite: 2026-02-28 22:00:00+00:00


#### CHECK DE CONTINUIDAD


In [11]:
expected_range = pd.date_range(
    start=df['datetime'].min(),
    end=df['datetime'].max(),
    freq='h'
)

missing = len(expected_range) - len(df)

print('Missing timestamps:', missing)

Missing timestamps: 0


#### ORDEN DE COLUMNAS


In [12]:
columns_order = [

    # TIEMPO
    'datetime', 'hour', 'dayofweek', 'month',

    # TARGET
    'price_spain',

    # DEMANDA
    'Prevista', 'Programada', 'Real', 'error_demanda',

    # CLIMA REAL
    'temperature', 'wind_speed',

    # GENERACION REAL
    'wind_generation',
    'solar_generation',
    'hidraulica_generation',
    'nuclear_generation',
    'ciclo_combinado_generation',
    'generacion_total',

    # SISTEMA
    'net_imports',
    'consumo_bombeo',
    'turbinacion_bombeo',
    'bombeo_neto',

    # FEATURES ESTRUCTURALES
    'renovable_ratio',
    'gas_ratio',
    'nuclear_ratio',
    'base_load',
    'base_vs_demanda',
    'gap_no_renovable',

    # EXTERNAS
    'gas_price',
    'co2_price',

    # PREVISIONES
    'wind_generation_forecast',
    'solar_generation_forecast',
    'temperature_forecast',
    'wind_speed_forecast',

    # TEMPORALES AVANZADAS
    'hour_sin', 'hour_cos',
    'day_sin', 'day_cos',
    'month_sin', 'month_cos',

    # LAGS
    'price_lag_1',
    'price_lag_24',

    # FLAGS
    'is_weekend',
    'is_holiday'
]

# Solo columnas que existen en el dataset
columns_order = [c for c in columns_order if c in df.columns]

df_model = df_model[[c for c in columns_order if c in df_model.columns]]

print('Columnas finales:', len(columns_order))
print(columns_order)

Columnas finales: 43
['datetime', 'hour', 'dayofweek', 'month', 'price_spain', 'Prevista', 'Programada', 'Real', 'error_demanda', 'temperature', 'wind_speed', 'wind_generation', 'solar_generation', 'hidraulica_generation', 'nuclear_generation', 'ciclo_combinado_generation', 'generacion_total', 'net_imports', 'consumo_bombeo', 'turbinacion_bombeo', 'bombeo_neto', 'renovable_ratio', 'gas_ratio', 'nuclear_ratio', 'base_load', 'base_vs_demanda', 'gap_no_renovable', 'gas_price', 'co2_price', 'wind_generation_forecast', 'solar_generation_forecast', 'temperature_forecast', 'wind_speed_forecast', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'price_lag_1', 'price_lag_24', 'is_weekend', 'is_holiday']


### 2. TRATAR NaNs


In [13]:
# ver NaNs
print(df.isna().sum())

datetime                        0
hour                            0
dayofweek                       0
month                           0
price_spain                   199
Prevista                      162
Programada                    162
Real                          183
error_demanda                 328
temperature                     1
wind_speed                      1
wind_generation                 0
solar_generation                0
hidraulica_generation           0
nuclear_generation              0
ciclo_combinado_generation      0
generacion_total                0
net_imports                     0
consumo_bombeo                  0
turbinacion_bombeo              0
bombeo_neto                     0
renovable_ratio                 0
gas_ratio                       0
nuclear_ratio                   0
base_load                       0
base_vs_demanda               162
gap_no_renovable                0
hour_sin                        0
hour_cos                        0
day_sin       

#### VARIABLE OBJETIVO (price)


In [14]:
df = df[df['price_spain'].notna()]

#### DEMANDA (forward fill)


In [15]:
# imputar demanda (variables base)
cols_demanda_base = ['Prevista', 'Programada', 'Real']
df[cols_demanda_base] = df[cols_demanda_base].ffill()

# recalcular variable derivada
df['error_demanda'] = df['Real'] - df['Prevista']

df = df.drop(columns=['base_vs_demanda'])
df['base_vs_demanda'] = df['base_load'] / df['Real']

#### LAGS


In [16]:
df = df.dropna(subset=['price_lag_1', 'price_lag_24'])

#### CLIMA REAL


In [17]:
df['temperature'] = df['temperature'].ffill().bfill()
df['wind_speed'] = df['wind_speed'].ffill().bfill()

#### PREVISIONES (forward fill y backward fill)


In [18]:
cols_prev = [
    'wind_generation_forecast',
    'solar_generation_forecast',
    'temperature_forecast',
    'wind_speed_forecast'
]

for col in cols_prev:
    if col in df.columns:
        df[col] = df[col].ffill().bfill()
        print(f'{col} - nulos restantes: {df[col].isna().sum()}')

wind_generation_forecast - nulos restantes: 0
solar_generation_forecast - nulos restantes: 0
temperature_forecast - nulos restantes: 0
wind_speed_forecast - nulos restantes: 0


#### LIMPIEZA FINAL


In [19]:
df = df.dropna().reset_index(drop=True)

print(df.isna().sum())
print(f'Total registros: {len(df)}')
print(f'Total columnas: {len(df.columns)}')

datetime                      0
hour                          0
dayofweek                     0
month                         0
price_spain                   0
Prevista                      0
Programada                    0
Real                          0
error_demanda                 0
temperature                   0
wind_speed                    0
wind_generation               0
solar_generation              0
hidraulica_generation         0
nuclear_generation            0
ciclo_combinado_generation    0
generacion_total              0
net_imports                   0
consumo_bombeo                0
turbinacion_bombeo            0
bombeo_neto                   0
renovable_ratio               0
gas_ratio                     0
nuclear_ratio                 0
base_load                     0
gap_no_renovable              0
hour_sin                      0
hour_cos                      0
day_sin                       0
day_cos                       0
month_sin                     0
month_co

### 3. Guardar


In [20]:
df.to_csv('dataset_final_clean.csv', index=False)
print('Dataset guardado como dataset_final_clean.csv')

Dataset guardado como dataset_final_clean.csv
